# SPCS Standalone SQL Server Connectivity Test

**Note: This Notebook should be run in an SPCS Container for testing to be valid**

This notebook provides modular components for testing connectivity to a standalone SQL Server instance from Snowflake Container Services (SPCS). Connectivity can be routed over a private link or a public network.

## Architecture:
- **Default (Private Link)**: SPCS Container → AWS PrivateLink / Azure Private Link → Standalone SQL Server
- **Public Network**: SPCS Container → Public Internet → Standalone SQL Server
- The Openflow runtime runs inside SPCS and requires an External Access Integration (EAI) with a network rule to reach the SQL Server endpoint. Private link is recommended for production workloads but not required.

## Quick Start Options:
1. **Basic Test**: Configure parameters and run network/auth tests
2. **Full Setup**: Use cells to set up PyPI access and SQL Server EAI as needed
3. **Custom Setup**: Pick and choose individual components as needed

## Components Available:
- **Configuration**: Set your SQL Server connection parameters
- **PyPI Setup**: Optional - enables installing mssql-python driver from PyPI
- **DNS Resolution Test**: Validates that the endpoint resolves correctly from within SPCS (useful for private link)
- **Network Test**: Tests basic network connectivity to SQL Server host
- **Authentication Test**: Tests SQL Server authentication and basic queries using the official Microsoft mssql-python driver
- **SQL Server EAI**: Optional - configures External Access Integration for your standalone SQL Server

## Openflow Connector Reference:
- The Openflow Connector for SQL Server uses JDBC URL format: `jdbc:sqlserver://<host>:<port>;encrypt=false`
- For standalone instances routed via private link, the `encrypt` property depends on your SQL Server TLS configuration
- See: https://docs.snowflake.com/en/user-guide/data-integration/openflow/connectors/sql-server/about

**Note: You must restart the Session after making EAI changes for them to take effect**

In [ ]:
# Standalone SQL Server Connectivity Test Configuration
# Update these parameters with your actual SQL Server connection details
# The host can be a private link endpoint DNS name or a publicly accessible hostname/IP

# User Configuration - UPDATE THIS
SQL_SERVER_HOST = "sqlserver.private.example.com"  # Private link endpoint, public DNS name, or IP address
SQL_SERVER_PORT = 1433
SQL_SERVER_DATABASE = "mydb"
SQL_SERVER_USER = "openflow_user"
SQL_SERVER_PASSWORD = "mypassword"
SQL_SERVER_ENCRYPT = False  # Set to True if your SQL Server requires TLS

# Openflow Connector JDBC URL (for reference when configuring the connector)
encrypt_str = "true" if SQL_SERVER_ENCRYPT else "false"
JDBC_URL = f"jdbc:sqlserver://{SQL_SERVER_HOST}:{SQL_SERVER_PORT};encrypt={encrypt_str}"

# This role will be used to create the EAI and other objects if necessary
IMPLEMENTATION_ROLE = "ACCOUNTADMIN"

# This role will be used by Openflow to run Connectors and perform your Data Engineering tasks
OPENFLOW_RUNTIME_ROLE = "OPENFLOWADMIN"

print(f"Configuration:")
print(f"SQL Server Host: {SQL_SERVER_HOST}")
print(f"SQL Server Port: {SQL_SERVER_PORT}")
print(f"SQL Server Database: {SQL_SERVER_DATABASE}")
print(f"SQL Server User: {SQL_SERVER_USER}")
print(f"Encrypt: {SQL_SERVER_ENCRYPT}")
print(f"\nOpenflow JDBC URL: {JDBC_URL}")
print("\nReady to test connectivity...")

## 1. PyPI Setup (Optional)

Run these cells if you need to install the mssql-python driver from PyPI. This creates the necessary network rules and External Access Integration for PyPI access.

**Skip this section if you already have mssql-python installed or have PyPI access configured.**

In [ ]:
-- Create Network Rule and External Access Integration for PyPI
-- Run this cell to enable installing Python packages from PyPI

USE ROLE {{IMPLEMENTATION_ROLE}};

CREATE OR REPLACE NETWORK RULE pypi_network_rule
  MODE = EGRESS
  TYPE = HOST_PORT
  VALUE_LIST = ('pypi.org', 'pypi.python.org', 'pythonhosted.org', 'files.pythonhosted.org');

CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION pypi_access_integration
  ALLOWED_NETWORK_RULES = (pypi_network_rule)
  ENABLED = true
  COMMENT = 'External Access Integration for PyPI package installation';

GRANT USAGE ON INTEGRATION pypi_access_integration TO ROLE {{IMPLEMENTATION_ROLE}};

SHOW EXTERNAL ACCESS INTEGRATIONS LIKE 'pypi_access_integration';

In [ ]:
-- Apply PyPI integration to this notebook
-- Run this after creating the PyPI integration above

ALTER NOTEBOOK EAI_STANDALONE_SQL
  SET EXTERNAL_ACCESS_INTEGRATIONS = ('pypi_access_integration', 'STANDALONE_SQL_EAI');

  -- Restart your Notebook session after applying an EAI

**NOTE: You will need to restart the Notebook session after applying an EAI**

In [ ]:
# Install Microsoft Python Driver for SQL Server (mssql-python)
# This is the official Microsoft driver - no ODBC dependency required
# Make sure PyPI access is configured first if you get connection errors
# You can run this cell twice; the first to install the driver, the second to confirm it is imported

try:
    import mssql_python
    print("mssql-python already available")
except ImportError:
    print("Installing mssql-python...")
    %pip install mssql-python
    print("mssql-python installed")

## 2. Connectivity Tests

These cells test connectivity to your standalone SQL Server. Run them in order to diagnose any connection issues.

**Note: If you need to install mssql-python, make sure to run the PyPI setup section first and restart your session.**

In [ ]:
# DNS Resolution Test
# Validates that the SQL Server endpoint resolves correctly from within SPCS
# Particularly useful for private link connections to confirm DNS routing

import socket

def test_dns_resolution(host):
    """Test DNS resolution of the SQL Server private endpoint"""
    try:
        print(f"Resolving DNS for: {host}")
        addresses = socket.getaddrinfo(host, None)
        unique_ips = set()
        for addr in addresses:
            ip = addr[4][0]
            unique_ips.add(ip)

        print(f"  Resolved to {len(unique_ips)} unique IP(s):")
        for ip in sorted(unique_ips):
            print(f"    - {ip}")

        for ip in unique_ips:
            parts = ip.split(".")
            if len(parts) == 4:
                first_octet = int(parts[0])
                second_octet = int(parts[1])
                is_private = (
                    first_octet == 10 or
                    (first_octet == 172 and 16 <= second_octet <= 31) or
                    (first_octet == 192 and second_octet == 168)
                )
                if is_private:
                    print(f"  [OK] {ip} is a private IP address (expected for PrivateLink)")
                else:
                    print(f"  [WARN] {ip} is a public IP address")
                    print(f"  This may indicate PrivateLink DNS is not configured correctly")

        return True
    except socket.gaierror as e:
        print(f"  [FAIL] DNS resolution failed: {e}")
        print(f"  Check that your private endpoint DNS is configured correctly")
        return False

print("=" * 50)
print("DNS RESOLUTION TEST")
print("=" * 50)

dns_result = test_dns_resolution(SQL_SERVER_HOST)

if dns_result:
    print(f"\nDNS resolution PASSED")
else:
    print(f"\nDNS resolution FAILED")
    print("Verify your PrivateLink endpoint and DNS configuration.")

In [ ]:
# Network Connectivity Test
# Tests basic TCP connectivity to SQL Server host and port via the private link

import socket
import time

def test_network_connectivity(host, port, description):
    """Test network connectivity to SQL Server host and port"""
    try:
        print(f"Testing {description}: {host}:{port}")

        start_time = time.time()
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(10)
        result = sock.connect_ex((host, port))
        elapsed = time.time() - start_time
        sock.close()

        if result == 0:
            print(f"  [OK] Network connection successful (latency: {elapsed*1000:.1f}ms)")
            return True
        else:
            print(f"  [FAIL] Network connection failed (error code: {result})")
            print(f"  This may indicate:")
            print(f"    - The SQL Server EAI network rule is missing or incorrect")
            print(f"    - The PrivateLink endpoint is not correctly configured")
            print(f"    - The SQL Server is not listening on port {port}")
            return False

    except socket.timeout:
        print(f"  [FAIL] Connection timed out after 10 seconds")
        print(f"  This typically means the private link route is not established")
        return False
    except Exception as e:
        print(f"  [FAIL] Network Error: {e}")
        return False

print("=" * 50)
print("NETWORK CONNECTIVITY TEST")
print("=" * 50)

network_result = test_network_connectivity(SQL_SERVER_HOST, SQL_SERVER_PORT, "SQL Server (via PrivateLink)")

if network_result:
    print(f"\nNetwork connectivity PASSED - SQL Server host is reachable via private link")
    print("You can proceed to test SQL Server authentication.")
else:
    print(f"\nNetwork connectivity FAILED")
    print("You need to configure a SQL Server External Access Integration (EAI).")
    print("See the SQL Server EAI Setup section below.")

In [ ]:
# SQL Server Authentication Test
# Tests authentication and basic queries using the official Microsoft mssql-python driver

import mssql_python

def test_sql_server_authentication():
    """Test SQL Server authentication and basic query"""
    try:
        print(f"Testing SQL Server Authentication")
        print(f"  Host: {SQL_SERVER_HOST}:{SQL_SERVER_PORT}")
        print(f"  Database: {SQL_SERVER_DATABASE}")
        print(f"  Encrypt: {SQL_SERVER_ENCRYPT}")

        encrypt_val = "yes" if SQL_SERVER_ENCRYPT else "no"
        connection_string = (
            f"SERVER=tcp:{SQL_SERVER_HOST},{SQL_SERVER_PORT};"
            f"DATABASE={SQL_SERVER_DATABASE};"
            f"UID={SQL_SERVER_USER};"
            f"PWD={SQL_SERVER_PASSWORD};"
            f"Encrypt={encrypt_val};"
            f"TrustServerCertificate=yes;"
        )

        conn = mssql_python.connect(connection_string)
        print(f"  [OK] Authentication successful")

        cursor = conn.cursor()
        cursor.execute("SELECT @@VERSION;")
        rows = cursor.fetchall()
        version = rows[0][0]
        print(f"  [OK] Database query successful")
        print(f"  SQL Server Version: {version.splitlines()[0]}")

        # Check if change tracking is enabled (required for Openflow connector)
        cursor.execute("""
            SELECT DB_NAME(database_id) AS database_name, is_auto_cleanup_on, retention_period, retention_period_units_desc
            FROM sys.change_tracking_databases;
        """)
        ct_dbs = cursor.fetchall()
        if ct_dbs:
            print(f"  [OK] Change Tracking enabled on {len(ct_dbs)} database(s):")
            for row in ct_dbs:
                print(f"     - {row[0]} (retention: {row[2]} {row[3]}, auto_cleanup: {row[1]})")
        else:
            print(f"  [WARN] Change Tracking is NOT enabled on any database")
            print(f"  The Openflow connector requires Change Tracking. Enable it with:")
            print(f"  ALTER DATABASE <db> SET CHANGE_TRACKING = ON (CHANGE_RETENTION = 2 DAYS, AUTO_CLEANUP = ON);")

        # List top 5 tables
        cursor.execute("""
            SELECT TOP 5 TABLE_SCHEMA, TABLE_NAME
            FROM INFORMATION_SCHEMA.TABLES
            WHERE TABLE_TYPE = 'BASE TABLE'
            ORDER BY TABLE_SCHEMA, TABLE_NAME;
        """)
        tables = cursor.fetchall()
        if tables:
            print(f"  Tables found ({len(tables)} shown):")
            for row in tables:
                print(f"     - {row[0]}.{row[1]}")

        conn.close()
        return True

    except Exception as e:
        print(f"  [FAIL] Database Error: {e}")
        print(f"  Troubleshooting:")
        print(f"    - Verify credentials are correct")
        print(f"    - Check SQL Server is configured for SQL authentication (mixed mode)")
        print(f"    - Verify firewall rules allow connections from the SPCS egress")
        print(f"    - If using TLS, set SQL_SERVER_ENCRYPT = True in config")
        return False


print("=" * 50)
print("SQL SERVER AUTHENTICATION TEST")
print("=" * 50)

auth_result = test_sql_server_authentication()

if auth_result:
    print(f"\nAuthentication PASSED")
    print(f"\nYour Openflow Connector JDBC URL should be:")
    print(f"  {JDBC_URL}")
    print(f"\nYou are ready to configure the Openflow Connector for SQL Server.")
else:
    print(f"\nAuthentication FAILED")
    print("Check SQL Server configuration, credentials, and network routing.")

## 3. SQL Server EAI Setup (Optional)

Run these cells if connectivity tests failed. This creates the necessary network rules and External Access Integration for your standalone SQL Server.

### Choosing the Network Rule Type

**Private Link (recommended for production):**
- Use `TYPE = PRIVATE_HOST_PORT` in the network rule
- Requires an existing outbound private link endpoint provisioned via `SYSTEM$PROVISION_PRIVATELINK_ENDPOINT`
- Traffic is routed through the private connectivity endpoint rather than over the public internet
- The private link endpoint must be in **"available"** status before creating the network rule

**Public Network:**
- Use `TYPE = HOST_PORT` in the network rule
- Traffic routes over the public internet
- Ensure your SQL Server firewall allows inbound connections from the SPCS egress IP range

In both cases, the EAI must be associated with the Openflow runtime (via the Openflow UI > Runtimes > External access integrations).

### Private Link Prerequisites (if using private connectivity)

The outbound private link endpoint must already be provisioned. Verify its status:

```sql
SELECT f.value
FROM TABLE(FLATTEN(input => parse_json(SYSTEM$GET_PRIVATELINK_ENDPOINTS_INFO()))) f;
```

Wait until the endpoint status is **"available"** before proceeding with the network rule below.

### References
- [SPCS Network Egress using Private Connectivity](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/additional-considerations-services-jobs#network-egress-using-private-connectivity)
- [Manage Private Endpoints (AWS)](https://docs.snowflake.com/en/user-guide/private-manage-endpoints-aws)
- [Private Azure Connectivity](https://docs.snowflake.com/en/developer-guide/external-network-access/creating-using-private-azure)
- [Openflow: Configure Allowed Domains](https://docs.snowflake.com/en/user-guide/data-integration/openflow/setup-openflow-spcs-sf-allow-list)

**Skip this section if your connectivity tests passed.**

In [ ]:
-- Create Network Rule for standalone SQL Server connectivity
-- Choose ONE of the two options below depending on your connectivity method

USE ROLE {{IMPLEMENTATION_ROLE}};

-- OPTION A: Private Link (recommended)
-- Uses PRIVATE_HOST_PORT to route traffic through the provisioned private link endpoint
-- Requires: SYSTEM$PROVISION_PRIVATELINK_ENDPOINT already completed and endpoint in "available" state
CREATE OR REPLACE NETWORK RULE STANDALONE_SQL_RULE
  MODE = EGRESS
  TYPE = PRIVATE_HOST_PORT
  VALUE_LIST = ('{{SQL_SERVER_HOST}}:{{SQL_SERVER_PORT}}');

-- OPTION B: Public Network (uncomment below and comment out Option A if using public connectivity)
-- CREATE OR REPLACE NETWORK RULE STANDALONE_SQL_RULE
--   MODE = EGRESS
--   TYPE = HOST_PORT
--   VALUE_LIST = ('{{SQL_SERVER_HOST}}:{{SQL_SERVER_PORT}}');

DESCRIBE NETWORK RULE STANDALONE_SQL_RULE;

In [ ]:
-- Create External Access Integration for standalone SQL Server
-- This EAI must be associated with your Openflow runtime via the Openflow UI

CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION STANDALONE_SQL_EAI
  ALLOWED_NETWORK_RULES = (
    STANDALONE_SQL_RULE
  )
  ENABLED = TRUE
  COMMENT = 'External Access Integration for standalone SQL Server connectivity';

GRANT USAGE ON INTEGRATION STANDALONE_SQL_EAI TO ROLE {{IMPLEMENTATION_ROLE}};
GRANT USAGE ON INTEGRATION STANDALONE_SQL_EAI TO ROLE {{OPENFLOW_RUNTIME_ROLE}};

SHOW EXTERNAL ACCESS INTEGRATIONS LIKE 'STANDALONE_SQL_EAI';

In [ ]:
-- Apply SQL Server EAI to this notebook
-- This enables the notebook to reach your standalone SQL Server
-- Include pypi_access_integration if you created it earlier

ALTER NOTEBOOK EAI_STANDALONE_SQL
  SET EXTERNAL_ACCESS_INTEGRATIONS = ('STANDALONE_SQL_EAI', 'pypi_access_integration');

  -- Restart your Notebook session after applying an EAI

## 4. Associate EAI with Openflow Runtime

After creating the EAI above, you must also associate it with your Openflow runtime for the connector to reach the SQL Server:

1. Navigate to the **Openflow** canvas in Snowsight
2. Select the **Runtimes** tab
3. For your runtime, click the **⋮** (More Options) menu
4. Select **External access integrations**
5. Add `STANDALONE_SQL_EAI` from the dropdown
6. Click **Save**

The change takes effect immediately without restarting the runtime.

### Openflow Connector Configuration Reference

When configuring the SQL Server connector parameters:

| Parameter | Value |
|-----------|-------|
| SQLServer Connection URL | `jdbc:sqlserver://<host>:<port>;encrypt=false` |
| SQLServer Username | Your SQL Server login |
| SQLServer Password | Your SQL Server password |
| Snowflake Authentication Strategy | `SNOWFLAKE_MANAGED_TOKEN` (for Snowflake deployments) |

For a standalone SQL Server instance, the connector discovers databases to replicate from the instance automatically.